# Which papers drop out when a retriever goes offline?

A hybrid search coordinator has already combined keyword and vector results. One worker withdraws its list. Which papers still reach the reader or the RAG context window?

Replay published BM25 and dense rankings for 16 SciFact queries. RankPatch maintains the running top *k*. A separate Python calculation rebuilds every score after each message. Change the query or the context size to see where the two retrievers agree.

[Open in Google Colab](https://colab.research.google.com/github/pgg898200-ship-it/rankpatch/blob/main/examples/retriever_outage.ipynb) · [RankPatch](https://github.com/pgg898200-ship-it/rankpatch) · [Try the browser demo](https://huggingface.co/spaces/pgg-engineering/rankpatch)

## Pick a query and a context size

Use a Python 3.11–3.13 kernel on a platform with a [release wheel](https://github.com/pgg898200-ship-it/rankpatch/releases/tag/v0.1.0). Colab's standard CPU runtime is sufficient. The notebook needs internet access for the wheel and a 28 kB fixture. The saved outputs use query `1008`, eight results per retriever and a three-paper context.

In [1]:
import subprocess
import sys
from importlib.metadata import version
from IPython.display import Markdown, display

release = "https://github.com/pgg898200-ship-it/rankpatch/releases/expanded_assets/v0.1.0"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                "--no-index", "--only-binary=:all:", "--find-links", release, "rankpatch==0.1.0"],
               check=True)
from rankpatch import Index

display(Markdown(f"Installed **RankPatch {version('rankpatch')}**."))

Installed **RankPatch 0.1.0**.

In [2]:
QUERY = 1008
DEPTH = 8  # 1–32 results per retriever
K = 3      # 1–DEPTH papers in the context

In [3]:
from collections import defaultdict
from hashlib import sha256
from urllib.request import urlopen

fixture_url = ("https://raw.githubusercontent.com/pgg898200-ship-it/rankpatch/"
               "81da2bf6dc5ac41a0c0de6e0e9f942b21a261e03/replay/scifact.txt")
with urlopen(fixture_url, timeout=30) as response:
    data = response.read()
assert sha256(data).hexdigest() == "2c01f62b83a0e38e782298fa85d63f179495ad132abd485e8574bd91f24a6891"

queries = defaultdict(dict)
for row in data.decode().splitlines():
    query, method, *ids = row.split()
    if method in ("bm25", "cosine_float"):
        queries[int(query)][method] = list(map(int, ids))
display(Markdown("Available query IDs: " + ", ".join(map(str, sorted(queries)))))
assert QUERY in queries and 1 <= K <= DEPTH <= 32, "Choose a listed query and 1 ≤ K ≤ DEPTH ≤ 32."

Available query IDs: 0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1008, 1009, 1011, 1012, 1014

## Withdraw a worker, then replay its last message

Both workers cast Borda points. First place earns `DEPTH` points, second place earns one fewer, and so on. Equal totals favor the smaller numeric paper ID. RankPatch takes compact integer IDs, so the coordinator maps the observed IDs in ascending order and maps the winners back on output.

An application decides when a worker has expired and sends a newer revision with an empty list. The older message that follows simulates a delayed network retry. The reference calculation keeps its own revision counters and sorts all active scores after **every** message.

In [4]:
from collections import Counter


def replay(query_id, depth, k):
    runs = queries[query_id]
    ballots = [runs[method][:depth] for method in ("bm25", "cosine_float")]
    papers = sorted(set(ballots[0]) | set(ballots[1]))
    dense_id = {paper: i for i, paper in enumerate(papers)}
    points = list(range(depth, 0, -1))
    index = Index(sources=2, universe=len(papers), rank_points=points, k=k)

    messages = [
        ("BM25 arrives",       0, 1, ballots[0]),
        ("Both online",        1, 1, ballots[1]),
        ("Dense offline",      1, 2, []),
        ("Late dense retry",   1, 1, ballots[1]),
        ("Dense restored",     1, 3, ballots[1]),
        ("BM25 offline",       0, 2, []),
    ]
    snapshots = {}
    revisions, active = [-1, -1], [[], []]
    for label, source, revision, ballot in messages:
        accepted = index.replace(source, revision, [dense_id[paper] for paper in ballot])
        fresh = revision > revisions[source]
        assert accepted == fresh
        if fresh:
            revisions[source] = revision
            active[source] = ballot
        scores = Counter()
        for ranked in active:
            scores.update(dict(zip(ranked, points)))
        expected = sorted(scores.items(), key=lambda hit: (-hit[1], hit[0]))[:k]
        actual = [(papers[i], score) for i, score in index.top()]
        assert actual == expected, (query_id, label, actual, expected)
        snapshots[label] = actual
    return snapshots

In [5]:
snap = replay(QUERY, DEPTH, K)
rows = ["| Delivery | Ordered context (paper ID · points) |",
        "|---|---|"]
for label, hits in snap.items():
    ranking = ", ".join(f"`{doc}` · {score}" for doc, score in hits)
    rows.append(f"| {label} | {ranking} |")
display(Markdown("\n".join(rows)))

| Delivery | Ordered context (paper ID · points) |
|---|---|
| BM25 arrives | `14105446` · 8, `19799455` · 7, `19343151` · 6 |
| Both online | `1982286` · 8, `14105446` · 8, `12315072` · 7 |
| Dense offline | `14105446` · 8, `19799455` · 7, `19343151` · 6 |
| Late dense retry | `14105446` · 8, `19799455` · 7, `19343151` · 6 |
| Dense restored | `1982286` · 8, `14105446` · 8, `12315072` · 7 |
| BM25 offline | `1982286` · 8, `12315072` · 7, `13048272` · 6 |

## How often does the context survive an outage?

A context can retain all its papers while moving a different paper into first place. Count the cases where the **set** survives and where the entire **sequence** survives. For the saved settings, switch to query `1000` to see the distinction in the delivery table.

In [6]:
traces = [replay(query, DEPTH, K) for query in sorted(queries)]
rows = ["| Missing worker | All papers retained | Same order | Mean papers retained |", "|---|---:|---:|---:|"]
for missing in ("Dense offline", "BM25 offline"):
    preserved, ordered, total = 0, 0, 0
    for trace in traces:
        both = [doc for doc, _ in trace["Both online"]]
        alone = [doc for doc, _ in trace[missing]]
        overlap = len(set(both) & set(alone))
        preserved += overlap == K
        ordered += alone == both
        total += overlap
    n = len(traces)
    rows.append(f"| {missing.split()[0]} | {preserved}/{n} | {ordered}/{n} | {total/n:.2f} of {K} |")
display(Markdown("\n".join(rows)))
display(Markdown(f"**{sum(map(len, traces))} snapshots matched a full score rebuild.**"))

| Missing worker | All papers retained | Same order | Mean papers retained |
|---|---:|---:|---:|
| Dense | 5/16 | 5/16 | 2.25 of 3 |
| BM25 | 6/16 | 3/16 | 2.19 of 3 |

**96 snapshots matched a full score rebuild.**

## What the replay measures

The ranked paper IDs come from recorded retrieval runs. The arrivals, withdrawals and retries are simulated. Overlap measures how much a downstream context changes. Deciding whether either context answers the query better requires relevance judgments. The 16 queries form an availability-selected subset, and the notebook measures no latency.

**Attribution.** The data are adapted from Timo Breuer’s [*SQLite is Enough. Lexical, Semantic, and Hybrid Search with scrydb*](https://arxiv.org/abs/2608.24060). RankPatch’s fixture retains the top 32 IDs for each selected query and method. This example selects BM25 and floating-point cosine results and truncates them to the requested depth. The derived data retain [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Notebook code follows the repository’s [Apache-2.0 license](https://github.com/pgg898200-ship-it/rankpatch/blob/main/LICENSE). [Pinned source data](https://huggingface.co/datasets/breuert/scrydb-eval/tree/ea5a009b6b0af4f2d164e95abca057090210db77) · [Fixture provenance](https://github.com/pgg898200-ship-it/rankpatch/blob/81da2bf6dc5ac41a0c0de6e0e9f942b21a261e03/replay/README.md)